### N-gram Language Model for Next Word Prediction

In [2]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import defaultdict, Counter

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
corpus_paragraph = """
The quick brown fox jumps over the lazy dog. The lazy dog barks loudly at the moon. The quick brown cat meows softly and quickly. A brown fox is faster than a lazy dog, always. The dog chases the cat. Machine learning is fascinating and can be applied to many problems, including natural language processing."""

sentences = sent_tokenize(corpus_paragraph.lower())
processed_corpus = [word_tokenize(sentence) for sentence in sentences]

print("Processed Corpus (sentences tokenized and lowercased):")
for i, tokens in enumerate(processed_corpus[:]):
    print(f"{i+1}. {tokens}")

Processed Corpus (sentences tokenized and lowercased):
1. ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
2. ['the', 'lazy', 'dog', 'barks', 'loudly', 'at', 'the', 'moon', '.']
3. ['the', 'quick', 'brown', 'cat', 'meows', 'softly', 'and', 'quickly', '.']
4. ['a', 'brown', 'fox', 'is', 'faster', 'than', 'a', 'lazy', 'dog', ',', 'always', '.']
5. ['the', 'dog', 'chases', 'the', 'cat', '.']
6. ['machine', 'learning', 'is', 'fascinating', 'and', 'can', 'be', 'applied', 'to', 'many', 'problems', ',', 'including', 'natural', 'language', 'processing', '.']


In [4]:
def generate_ngrams(tokens_list, n):
    ngrams = []
    for sentence_tokens in tokens_list:
        if len(sentence_tokens) >= n:
            for i in range(len(sentence_tokens) - n + 1):
                ngrams.append(tuple(sentence_tokens[i : i + n]))
    return ngrams

unigrams = generate_ngrams(processed_corpus, 1)
bigrams = generate_ngrams(processed_corpus, 2)
trigrams = generate_ngrams(processed_corpus, 3)

unigram_counts = Counter(word for sentence_tokens in processed_corpus for word in sentence_tokens)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

trigram_probabilities = defaultdict(lambda: defaultdict(int))
for t_gram in trigram_counts:
    prefix = t_gram[0:2]
    next_word = t_gram[2]
    trigram_probabilities[prefix][next_word] = trigram_counts[t_gram]

for prefix in trigram_probabilities:
    total_count = sum(trigram_probabilities[prefix].values())
    for next_word in trigram_probabilities[prefix]:
        trigram_probabilities[prefix][next_word] /= total_count

print("Sample Trigram Probabilities")
if trigram_probabilities:
    first_prefix = next(iter(trigram_probabilities))
    print(f"Context {first_prefix}: {dict(trigram_probabilities[first_prefix])}")
else:
    print("No trigram probabilities generated (corpus might be too short).")

Sample Trigram Probabilities
Context ('the', 'quick'): {'brown': 1.0}


In [5]:
def predict_next_word(context_words):
    context = tuple(word.lower() for word in context_words)

    if len(context) >= 2:
        prefix = context[-2:]
        if prefix in trigram_probabilities:
            return max(trigram_probabilities[prefix], key=trigram_probabilities[prefix].get)

    if len(context) >= 1:
        prefix_word = context[-1]
        possible_next_words = defaultdict(int)
        for b_gram, count in bigram_counts.items():
            if b_gram[0] == prefix_word:
                possible_next_words[b_gram[1]] += count
        if possible_next_words:
            return max(possible_next_words, key=possible_next_words.get)

    if unigram_counts:
        return unigram_counts.most_common(1)[0][0]

    return "<UNK>"

test_contexts = [
    ["The", "quick"],
    ["lazy", "dog"],
    ["brown", "fox"],
    ["the", "cat"],
    ["a"],
    ["language", "processing"]
]

print("\nNext Word Predictions:")
for context in test_contexts:
    predicted_word = predict_next_word(context)
    print(f"Context: {' '.join(context)} -> Predicted next word: {predicted_word}")


Next Word Predictions:
Context: The quick -> Predicted next word: brown
Context: lazy dog -> Predicted next word: .
Context: brown fox -> Predicted next word: jumps
Context: the cat -> Predicted next word: .
Context: a -> Predicted next word: brown
Context: language processing -> Predicted next word: .
